# PDF 고시 전문 -> JSON

건강기능식품의 표시기준 고시 전문 PDF를 페이지별 텍스트와 메타데이터가 포함된 JSON으로 변환합니다.

주의: 스캔 이미지 PDF는 텍스트가 추출되지 않을 수 있어 OCR이 필요합니다.

In [ ]:
# 최초 1회만 실행합니다.
# %pip install pdfplumber -q

Note: you may need to restart the kernel to use updated packages.


In [7]:
from pathlib import Path
from datetime import datetime
import json
import re
import pdfplumber

# [입력 방법]
# PDF가 이 노트북의 작업 폴더 안에 있으면 None으로 두세요(하위 폴더까지 자동 탐색).
# 다른 폴더 PDF는 예시처럼 절대 경로를 입력하세요: Path(r'C:\Users\홍길동\Downloads\문서.pdf')
# Windows 경로는 r'...' 또는 슬래시(/)를 사용하고, 반드시 .pdf 파일을 지정하세요.
# 파일명이 여러 개 일치하면 오류가 나므로 INPUT_PDF에 원하는 파일을 직접 지정하세요.
INPUT_PDF = Path(r"data\law\건강기능식품의+기준+및+규격+일부개정고시(제2026-43호,+2026.6.11.).pdf")
OUTPUT_JSON = Path('건강기능식품의+기준+및+규격+일부개정고시(제2026-43호,+2026.6.11.).pdf')
TARGET_NAME = '건강기능식품의+기준+및+규격+일부개정고시(제2026-43호,+2026.6.11.).pdf'
# 입력 PDF를 바꿀 때 아래 메타데이터도 실제 문서에 맞게 수정하세요.
DOCUMENT_TITLE = '건강기능식품의 기준 및 규격'
ISSUING_AUTHORITY = '식품의약품안전처'
NOTICE_NUMBER = '고시 제2026-43호'
PUBLISHED_DATE = '2026-06-11'
ROOT = Path.cwd()

def find_input_pdf(value):
    if value is not None:
        path = Path(value).expanduser().resolve()
        if not path.is_file(): raise FileNotFoundError(f'입력 PDF를 찾을 수 없습니다: {path}')
        if path.suffix.lower() != '.pdf': raise ValueError('입력 파일은 PDF여야 합니다.')
        return path
    candidates = [p for p in ROOT.rglob('*.pdf') if p.name == TARGET_NAME]
    if not candidates: candidates = [p for p in ROOT.rglob('*.pdf') if '2025-61호' in p.name and '표시기준' in p.name]
    if not candidates: raise FileNotFoundError('대상 PDF를 찾지 못했습니다. INPUT_PDF에 경로를 직접 지정하세요.')
    if len(candidates) > 1:
        raise RuntimeError('일치하는 PDF가 여러 개입니다. INPUT_PDF에 하나를 직접 지정하세요.\n' + '\n'.join(map(str, candidates)))
    return candidates[0].resolve()

PDF_PATH = find_input_pdf(INPUT_PDF)
# PDF 결과 JSON은 원본 PDF와 동일한 폴더에 저장합니다.
# 원본 PDF와 같은 폴더에, PDF 파일명만 .json으로 바꾸어 저장합니다.
# 예: 문서.pdf -> 문서.json (사용자가 지정한 PDF 제목을 그대로 유지)
OUTPUT_PATH = PDF_PATH.with_suffix('.json')
print(f'입력 PDF: {PDF_PATH}')
print(f'출력 JSON: {OUTPUT_PATH}')

입력 PDF: C:\Users\Playdata\Desktop\mle-01-final-team3\data\law\건강기능식품의+기준+및+규격+일부개정고시(제2026-43호,+2026.6.11.).pdf
출력 JSON: C:\Users\Playdata\Desktop\mle-01-final-team3\data\law\건강기능식품의+기준+및+규격+일부개정고시(제2026-43호,+2026.6.11.).json


In [8]:
def normalize_line(value: str) -> str:
    value = value.replace('\r\n', '\n').replace('\r', '\n')
    return re.sub(r'[ \t]+', ' ', value).strip()

pages = []
with pdfplumber.open(PDF_PATH) as pdf:
    for page_number, page in enumerate(pdf.pages, start=1):
        raw_text = page.extract_text(x_tolerance=2, y_tolerance=3) or ''
        lines = [normalize_line(line) for line in raw_text.split('\n')]
        lines = [line for line in lines if line]
        pages.append({'page_number': page_number, 'text': '\n'.join(lines), 'lines': lines, 'character_count': len(''.join(lines))})
full_text = '\n'.join(page['text'] for page in pages)
replacement_count = full_text.count('�')
result = {'document': {'title': DOCUMENT_TITLE, 'issuing_authority': ISSUING_AUTHORITY, 'notice_number': NOTICE_NUMBER, 'published_date': PUBLISHED_DATE, 'source_pdf': str(PDF_PATH), 'page_count': len(pages), 'extraction_method': 'pdfplumber', 'extracted_at': datetime.now().astimezone().isoformat(), 'replacement_character_count': replacement_count, 'quality_warning': 'PDF 텍스트에 대체문자가 포함됨' if replacement_count else None}, 'pages': pages}
print(f'페이지 수: {len(pages):,}, 문자 수: {len(full_text):,}')
if not full_text.strip(): print('경고: 추출된 텍스트가 없습니다. 스캔 PDF인지 확인하고 OCR을 사용하세요.')
elif replacement_count: print(f'경고: 대체문자(�) {replacement_count:,}개가 포함되었습니다.')

페이지 수: 22, 문자 수: 12,112


In [9]:
with OUTPUT_PATH.open('w', encoding='utf-8') as file: json.dump(result, file, ensure_ascii=False, indent=2)
with OUTPUT_PATH.open(encoding='utf-8') as file: saved = json.load(file)
assert saved['document']['page_count'] == len(saved['pages'])
assert all(page['page_number'] == index for index, page in enumerate(saved['pages'], start=1))
print(f'저장 완료: {OUTPUT_PATH}')

저장 완료: C:\Users\Playdata\Desktop\mle-01-final-team3\data\law\건강기능식품의+기준+및+규격+일부개정고시(제2026-43호,+2026.6.11.).json


## CSV -> JSON

CSV의 첫 행을 JSON 키로 사용해 각 행을 레코드로 저장합니다.

In [77]:
import csv

# [CSV 입력 방법]
# 작업 폴더 아래에 CSV가 있으면 None으로 두면 하위 폴더까지 자동 탐색합니다.
# 다른 폴더의 CSV는 예: Path(r'C:\\Users\\Hong\\Downloads\\file.csv')
# Windows 경로는 r'...' 또는 슬래시(/)를 사용하고, CSV 파일 자체의 경로를 입력하세요.
# 같은 이름의 CSV가 여러 개면 INPUT_CSV에 원하는 파일 하나를 직접 지정하세요.
INPUT_CSV = None
CSV_NAME = 'I1250_261123.csv'
CSV_OUTPUT_NAME = 'I1250_261123.json'

CSV_TITLE_BY_PREFIX = {
    '건강기능식품_개별인정형_정보': '건강기능식품 개별인정형 정보',
    '건강기능식품_기능성_원료인정현황': '건강기능식품 기능성 원료인정현황',
    '건강기능식품_영양DB': '건강기능식품 영양DB',
    '건강기능식품_품목분류정보': '건강기능식품 품목분류정보',
    'I0030_': '건강기능식품 품목제조 신고사항 현황',
    'C002_': '식품(첨가물) 품목제조보고 원재료',
    'I1250_': '식품(첨가물) 품목제조보고',
}

def resolve_csv_title(path):
    for prefix, title in CSV_TITLE_BY_PREFIX.items():
        if path.stem.startswith(prefix):
            return title
    raise ValueError(f'등록되지 않은 CSV 데이터셋입니다: {path.name}')

def find_input_csv(value):
    if value is not None:
        path = Path(value).expanduser().resolve()
        if not path.is_file(): raise FileNotFoundError(f'CSV not found: {path}')
        if path.suffix.lower() != '.csv': raise ValueError('Input file must be CSV.')
        return path
    candidates = list(ROOT.rglob(CSV_NAME))
    if not candidates: raise FileNotFoundError('CSV not found. Set INPUT_CSV to the file path.')
    if len(candidates) > 1: raise RuntimeError('Multiple CSV files found. Set INPUT_CSV to one file.')
    return candidates[0].resolve()

CSV_PATH = find_input_csv(INPUT_CSV)
# CSV 결과 JSON도 원본 CSV와 동일한 폴더에 저장합니다.
CSV_OUTPUT_PATH = CSV_PATH.parent / CSV_OUTPUT_NAME

def read_csv_records(path):
    # utf-8-sig는 Excel의 UTF-8 BOM CSV를 처리하며, cp949도 보조로 지원합니다.
    try:
        with path.open('r', encoding='utf-8-sig', newline='') as file:
            return list(csv.DictReader(file))
    except UnicodeDecodeError:
        with path.open('r', encoding='cp949', newline='') as file:
            return list(csv.DictReader(file))

records = read_csv_records(CSV_PATH)
if not records: raise ValueError('CSV has no data rows.')
if any(None in record for record in records): raise ValueError('CSV columns do not match row lengths.')
csv_result = {'document': {'title': resolve_csv_title(CSV_PATH), 'source_csv': str(CSV_PATH), 'record_count': len(records), 'column_count': len(records[0]), 'columns': list(records[0].keys()), 'extraction_method': 'csv.DictReader', 'extracted_at': datetime.now().astimezone().isoformat()}, 'records': records}
with CSV_OUTPUT_PATH.open('w', encoding='utf-8') as file: json.dump(csv_result, file, ensure_ascii=False, indent=2)
with CSV_OUTPUT_PATH.open(encoding='utf-8') as file: saved_csv = json.load(file)
assert saved_csv['document']['record_count'] == len(saved_csv['records'])
print(f'저장 완료: {CSV_OUTPUT_PATH}')

저장 완료: C:\Users\Playdata\Desktop\mle-01-final-team3\data\02_일반식품-20261123T025250Z-1-001\02_일반식품\원본데이터\I1250\I1250_261123.json
